# Day 9 v2 — Silver Transformation: All Blob Entities (For Loop)

**Source:** `/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime/`
**Sink:** `/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime/` (Delta)

**What changed from v1:**
- Added entity config list for all blob entities (charging_sessions + maintenance_events)
- Wrapped the same v1 steps inside a `for` loop
- Full overwrite mode (incremental MERGE is v3)

**Blob Bronze entities covered:**
| Entity | Source folder | Natural Key |
|---|---|---|
| charging_sessions | `realtime/charging_sessions/` | `session_id` |
| maintenance_events | `realtime/maintenance_events/` | `event_id` |

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Constants
BRONZE_REALTIME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/realtime"
SILVER_REALTIME = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/realtime"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze realtime : {BRONZE_REALTIME}")
print(f"Silver realtime : {SILVER_REALTIME}")
print(f"Run time        : {RUN_TS}")

In [ ]:
# Cell 3 — Entity config list
# Same pattern as Day 8 v2 — entity_name, natural_key, cast map.
# Bronze for blob entities is CSV (not JSON), so no explode step needed.

ENTITIES = [
    {
        "entity_name": "charging_sessions",
        "natural_key": "session_id",
        "cdc_field"  : "updated_at",
        "cast": {
            "session_id"      : "string",
            "vehicle_id"      : "string",
            "charger_id"      : "string",
            "customer_id"     : "string",
            "station_id"      : "string",
            "start_time"      : "timestamp",
            "end_time"        : "timestamp",
            "energy_kwh"      : "decimal(10,4)",
            "duration_minutes": "integer",
            "status"          : "string",
            "cost_aud"        : "decimal(10,2)",
            "created_at"      : "timestamp",
            "updated_at"      : "timestamp",
        }
    },
    {
        "entity_name": "maintenance_events",
        "natural_key": "event_id",
        "cdc_field"  : "updated_at",
        "cast": {
            "event_id"       : "string",
            "charger_id"     : "string",
            "station_id"     : "string",
            "event_type"     : "string",
            "description"    : "string",
            "technician_id"  : "string",
            "status"         : "string",
            "scheduled_date" : "date",
            "completed_date" : "date",
            "created_at"     : "timestamp",
            "updated_at"     : "timestamp",
        }
    },
]

print(f"Entities configured: {len(ENTITIES)}")
for e in ENTITIES:
    print(f"  {e['entity_name']:<25} natural_key={e['natural_key']}")

In [ ]:
# Cell 4 — For loop: transform all blob entities
# Each iteration:
#   1. Read Bronze CSV (all files under realtime/<entity>/)
#   2. Cast columns
#   3. Trim whitespace
#   4. Add audit columns
#   5. Deduplicate on natural key
#   6. Write to Silver Delta (overwrite)

results = []

for entity in ENTITIES:
    entity_name = entity["entity_name"]
    natural_key = entity["natural_key"]
    cdc_field   = entity["cdc_field"]
    cast_map    = entity["cast"]

    bronze_path = f"{BRONZE_REALTIME}/{entity_name}"
    silver_path = f"{SILVER_REALTIME}/{entity_name}"

    print(f"Processing: {entity_name} ...", end=" ")

    try:
        # Step 1: Read Bronze CSV
        raw_df = (
            spark.read
            .option("header", "true")
            .option("inferSchema", "true")
            .csv(bronze_path)
        )
        bronze_rows = raw_df.count()

        # Step 2: Cast columns
        cast_exprs = [
            F.col(c).cast(t).alias(c)
            for c, t in cast_map.items()
            if c in raw_df.columns
        ]
        typed_df = raw_df.select(cast_exprs)

        # Step 3: Trim string columns
        string_cols = [c for c, t in typed_df.dtypes if t == "string"]
        for col in string_cols:
            typed_df = typed_df.withColumn(col, F.trim(F.col(col)))

        # Step 4: Add audit columns
        audited_df = (
            typed_df
            .withColumn("silver_ingested_at", F.lit(RUN_TS).cast("timestamp"))
            .withColumn("silver_load_type",   F.lit("full"))
            .withColumn("silver_pipeline",    F.lit("pl_silver_blob_all_entities_v2"))
        )

        # Step 5: Deduplicate
        window = Window.partitionBy(natural_key).orderBy(F.col(cdc_field).desc())
        deduped_df = (
            audited_df
            .withColumn("_row_num", F.row_number().over(window))
            .filter(F.col("_row_num") == 1)
            .drop("_row_num")
        )
        silver_rows = deduped_df.count()

        # Step 6: Write to Silver
        (
            deduped_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(silver_path)
        )

        print(f"OK  (bronze={bronze_rows} -> silver={silver_rows})")
        results.append({"entity": entity_name, "status": "succeeded",
                        "bronze_rows": bronze_rows, "silver_rows": silver_rows, "error": None})

    except Exception as e:
        print(f"FAILED")
        results.append({"entity": entity_name, "status": "failed",
                        "bronze_rows": 0, "silver_rows": 0, "error": str(e)})

print("\nAll entities done.")

In [ ]:
# Cell 5 — Run summary

succeeded = [r for r in results if r["status"] == "succeeded"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 65)
print("SILVER BLOB TRANSFORMATION v2 — RUN SUMMARY")
print("=" * 65)
print(f"  run_timestamp : {RUN_TS}")
print(f"  succeeded     : {len(succeeded)}")
print(f"  failed        : {len(failed)}")
print()

for r in results:
    tag = "[OK]  " if r["status"] == "succeeded" else "[FAIL]"
    print(f"  {tag} {r['entity']:<25} bronze={r['bronze_rows']:>8}  silver={r['silver_rows']:>8}")
    if r["error"]:
        print(f"         Error: {r['error']}")

if failed:
    raise Exception(f"{len(failed)} entity(ies) failed — check output above.")
else:
    print(f"\nAll {len(succeeded)} entities written to Silver successfully.")